<a href="https://colab.research.google.com/github/asirabm/cctv-video-anomaly-detection/blob/main/05_lstm_model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, classification_report

In [20]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch version:", torch.__version__)
print("Computing device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.14.1+cpu
Computing device: cpu


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
project_path="/content/drive/MyDrive/UCF_Crime_Project"

In [4]:
prepared_path=os.path.join(project_path,"prepared_data")

In [5]:
train_data=np.load(os.path.join(prepared_path,'train_data.npz'))
vali_data=np.load(os.path.join(prepared_path,'validation_data.npz'))

In [6]:
train_data['X'][2][0][4]

np.float32(0.034236778)

In [7]:
X_train,y_train=train_data['X'],train_data['y']
x_val,y_val=vali_data['X'],vali_data['y']

In [8]:
print("Training:",X_train.shape,y_train.shape)
print("Validation:",x_val.shape,y_val.shape)

Training: (1288, 16, 2048) (1288,)
Validation: (322, 16, 2048) (322,)


**Convert Numpy data into PyTorch tensors**

In [9]:
X_train_tensor=torch.tensor(X_train,dtype=torch.float32)
y_train_tensor=torch.tensor(y_train,dtype=torch.long)

In [10]:
x_val_tensor=torch.tensor(x_val,dtype=torch.float32)
y_val_tensor=torch.tensor(y_val,dtype=torch.long)

In [11]:
print("X_train_tensor shape:",X_train_tensor.shape )
print("y_train_tensor shape:",y_train_tensor.shape )
print("x_validation_tensor shape:",x_val_tensor.shape )
print("y_validation_tensor shape:",y_val_tensor.shape )

X_train_tensor shape: torch.Size([1288, 16, 2048])
y_train_tensor shape: torch.Size([1288])
x_validation_tensor shape: torch.Size([322, 16, 2048])
y_validation_tensor shape: torch.Size([322])


**Create DataLoaders for divides the dataset into smalleer batches and supply them to model**

In [12]:
#First Create Dataset as feature and label combine
train_dataset=TensorDataset(X_train_tensor,y_train_tensor)
vali_dataset=TensorDataset(x_val_tensor,y_val_tensor)

In [13]:
#create dataloader
batch_size=32
train_loader=DataLoader(train_dataset,batch_size=batch_size,shuffle=True)
vali_loader=DataLoader(vali_dataset,batch_size=batch_size,shuffle=False)

In [14]:
print("Total training batches: ", len(train_loader))
print("Total validation batches: ", len(vali_loader))

Total training batches:  41
Total validation batches:  11


In [15]:
features, labels = next(iter(train_loader))

print("Batch feature shape:", features.shape)
print("Batch label shape:", labels.shape)

Batch feature shape: torch.Size([32, 16, 2048])
Batch label shape: torch.Size([32])


**Implement the LSTM model**

In [22]:
class LSTMforVideo(nn.Module):
  def __init__(self, input_size=2048,hidden_size=128,num_class=2):
    super().__init__()

    self.lstm=nn.LSTM(
        input_size=input_size,
        hidden_size=hidden_size,
        num_layers=1,
        batch_first=True
    )
    self.fc=nn.Linear(hidden_size,num_class)
  def forward(self,x):
    output,(hidden,cell)=self.lstm(x)
    final_output=output[:,-1,:]
    prediction=self.fc(final_output)
    return prediction




**create the model**

In [23]:
lstm_model=LSTMforVideo().to(device)

In [24]:
print(lstm_model)

LSTMforVideo(
  (lstm): LSTM(2048, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=2, bias=True)
)


**test the model's output**

In [29]:
next(iter(train_loader))

[tensor([[[6.9151e-02, 3.8583e-02, 7.6958e-02,  ..., 6.3042e-03,
           0.0000e+00, 1.1902e-02],
          [0.0000e+00, 1.2305e-02, 2.7349e-02,  ..., 0.0000e+00,
           0.0000e+00, 1.3401e-01],
          [7.9963e-03, 7.4614e-02, 7.3359e-02,  ..., 3.2350e-02,
           0.0000e+00, 2.4565e-01],
          ...,
          [1.9359e-01, 0.0000e+00, 0.0000e+00,  ..., 5.3980e-03,
           0.0000e+00, 0.0000e+00],
          [5.5137e-02, 6.3526e-02, 1.8568e-02,  ..., 8.4267e-03,
           0.0000e+00, 2.8563e-02],
          [1.3328e-02, 3.4416e-02, 1.1331e-02,  ..., 0.0000e+00,
           0.0000e+00, 1.9537e-03]],
 
         [[8.5276e-02, 1.6875e-02, 8.8537e-01,  ..., 0.0000e+00,
           0.0000e+00, 7.0954e-03],
          [3.5235e-02, 0.0000e+00, 9.4057e-01,  ..., 0.0000e+00,
           0.0000e+00, 3.4511e-02],
          [4.6195e-02, 7.7450e-03, 1.1595e+00,  ..., 0.0000e+00,
           0.0000e+00, 1.0684e-01],
          ...,
          [9.5205e-02, 1.9389e-02, 1.6093e+00,  ..., 0.000

In [30]:
sample_features,sample_labels=next(iter(train_loader))

In [31]:
sample_features=sample_features.to(device)

In [33]:
with torch.no_grad():
  sample_output=lstm_model(sample_features)

In [34]:
print(sample_output)

tensor([[ 2.7843e-02,  6.8050e-02],
        [ 1.9017e-01,  8.8970e-02],
        [-1.0508e-01,  2.5516e-02],
        [ 9.3044e-02,  4.2810e-02],
        [-1.0301e-01,  6.0106e-05],
        [-4.8439e-02, -6.5340e-02],
        [ 9.1191e-02, -7.9226e-02],
        [ 2.6165e-02,  4.4676e-02],
        [-5.5297e-02,  9.5766e-03],
        [-8.9969e-02,  1.7449e-01],
        [ 9.2159e-02,  1.5217e-01],
        [-5.8176e-02,  9.2701e-02],
        [-2.1393e-01,  2.5260e-02],
        [-5.2596e-02,  4.4645e-02],
        [-8.9091e-02,  9.1550e-02],
        [-1.6936e-02,  8.6019e-02],
        [-1.0092e-01,  8.0934e-02],
        [-6.9788e-02,  1.7437e-01],
        [ 1.3476e-02, -4.0567e-03],
        [-5.8391e-02, -5.7603e-02],
        [-1.2000e-01,  2.2487e-02],
        [-6.9268e-02,  2.6373e-03],
        [-1.7052e-01, -9.2672e-02],
        [-5.1034e-02, -6.1478e-03],
        [ 2.3162e-01,  3.2072e-02],
        [-1.0899e-01, -2.6436e-01],
        [-9.4966e-02,  8.0092e-02],
        [ 5.5582e-02, -1.566

In [35]:
print("Input shape:", sample_features.shape)
print("Output shape:", sample_output.shape)

Input shape: torch.Size([32, 16, 2048])
Output shape: torch.Size([32, 2])
